# 01. Data Cleaning

This notebook performs data cleaning and quality assessment on the raw datasets.
The cleaned datasets generated here are used as input for the subsequent imputation steps.

## 1. Import Libraries

In [15]:
import pandas as pd
import numpy as np

## 2. Load Data

### 2.1. Fronds Dataset

Each row represents one individual frond, with morphometric measurements (length, width, leaf area) and the total sorus count.

In [16]:
df_fronds_raw = pd.read_csv("../data/raw/fronds.csv")
df_fronds_raw.head()



,ID_Individuo,L_Fronda_cm,A_Fronda_cm,N_soros,Area_foliar_cm2
0,H01,63.0,8.4,1656.0,280.0
1,H02,42.0,5.8,1554.0,130.0
2,H03,68.5,8.1,1728.0,266.0
3,H04,66.0,7.1,1826.0,216.0
4,H05,60.7,6.0,1456.0,194.0


### 2.2. Sori Dataset

Each row represents one pair of pinnae (leaflets) at a given position along the frond, identified by `Id_fila` (`row_id`). Sori were counted on only one side of each pinna pair, based on an assumption of bilateral symmetry, and the recorded value was doubled to represent the full pair. Multiple rows belong to the same individual `ID_individuo` (`individual_id`), so this dataset is at a finer granularity than Fronds.

In [17]:
df_sori_raw = pd.read_csv("../data/raw/sori.csv")
df_sori_raw.head()

,ID_Individuo,Id_fila,No. soros por fila
0,H01,86,0.0
1,H01,85,0.0
2,H01,84,0.0
3,H01,83,0.0
4,H01,82,0.0


## 3. Initial Data Inspection

Before cleaning the data, we run a quick audit on both raw datasets to check their shape, memory footprint, duplicate rows, missing values, and any formatting issues in the column names. The `audit_data` function below reports these metrics and will be reused throughout this notebook.

In [18]:
def audit_data(df):

    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Duplicates: {df.duplicated().sum()}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    columns_with_spaces = [col for col in df.columns if col != col.strip()]
    print(f"Columns with leading/trailing spaces: {columns_with_spaces if columns_with_spaces else 'None'}")

    summary = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing": df.isnull().sum(),
        "% Missing": (df.isnull().mean() * 100).round(2),
        "Unique": df.nunique(),
        "Memory (KB)": (df.memory_usage(deep=True)[1:] / 1024).round(2)
    })

    return summary

In [19]:
audit_data(df_fronds_raw)

Shape: 24 rows × 5 columns
Duplicates: 0
Memory: 0.00 MB
Columns with leading/trailing spaces: ['A_Fronda_cm ']


,Data Type,Missing,% Missing,Unique,Memory (KB)
ID_Individuo,object,0,0.00,24,1.41
L_Fronda_cm,float64,0,0.00,23,0.19
A_Fronda_cm,float64,0,0.00,16,0.19
N_soros,float64,1,4.17,23,0.19
Area_foliar_cm2,float64,0,0.00,20,0.19


In [20]:
audit_data(df_sori_raw)

Shape: 1,921 rows × 3 columns
Duplicates: 0
Memory: 0.14 MB
Columns with leading/trailing spaces: ['No. soros por fila ']


,Data Type,Missing,% Missing,Unique,Memory (KB)
ID_Individuo,object,0,0.00,24,112.56
Id_fila,int64,0,0.00,103,15.01
No. soros por fila,float64,86,4.48,32,15.01


**Summary of findings**

- Neither dataset has duplicate rows.
- Both datasets have column names with formatting issues: `A_Fronda_cm ` (Fronds) has a trailing space, and `No. soros por fila ` (Sori) contains a period, an internal space, and a trailing space. All column names are also currently in Spanish.
- The Fronds dataset has 1 missing value (4.17%) in `N_soros`; the Sori dataset has 86 missing values (4.48%) in the sori-count column. These will be addressed in the imputation stage of the pipeline, this notebook only standardizes structure and format.


## 4. Data Cleaning

This section standardizes both datasets: column names are cleaned and translated to English, data types are corrected, and basic validation checks are run before exporting the cleaned datasets for the next stage of the pipeline.

### 4.1 Column Name Standardization

We strip leading/trailing whitespace, convert to lowercase, and replace internal spaces with underscores. Column names are then translated to English for consistency with the rest of the project.

In [21]:
df_fronds_clean = df_fronds_raw.copy()
df_sori_clean = df_sori_raw.copy()

df_fronds_clean.columns = df_fronds_clean.columns.str.strip().str.lower().str.replace(' ', '_')
df_sori_clean.columns = df_sori_clean.columns.str.strip().str.lower().str.replace(' ', '_')

df_fronds_clean = df_fronds_clean.rename(columns={
    'id_individuo': 'individual_id',
    'l_fronda_cm': 'frond_length_cm',
    'a_fronda_cm': 'frond_width_cm',
    'n_soros': 'n_sori_total',
    'area_foliar_cm2': 'leaf_area_cm2',
})

df_sori_clean = df_sori_clean.rename(columns={
    'id_individuo': 'individual_id',
    'id_fila': 'row_id',
    'no._soros_por_fila': 'n_sori_per_row',
})


Quick check of the result:

In [22]:
df_sori_clean.head()

,individual_id,row_id,n_sori_per_row
0,H01,86,0.0
1,H01,85,0.0
2,H01,84,0.0
3,H01,83,0.0
4,H01,82,0.0


In [23]:
df_fronds_clean.head()

,individual_id,frond_length_cm,frond_width_cm,n_sori_total,leaf_area_cm2
0,H01,63.0,8.4,1656.0,280.0
1,H02,42.0,5.8,1554.0,130.0
2,H03,68.5,8.1,1728.0,266.0
3,H04,66.0,7.1,1826.0,216.0
4,H05,60.7,6.0,1456.0,194.0


Both datasets now have clean, English, snake_case column names, consistent with the rest of the project. `df_fronds_clean` and `df_sori_clean` share the `individual_id` key, which will be used to merge both datasets in later stages of the pipeline.

### 4.2 Data Type Conversion

As reported by `audit_data` in Section 3, `n_sori_total` (Fronds) and `n_sori_per_row` (Sori) are stored as `float64` instead of an integer type, even though they represent counts. This is expected: pandas automatically upcasts an integer column to `float64` whenever it contains at least one missing value, since the standard NumPy integer dtype cannot represent `NaN`.

Both columns still have missing values at this stage and will not be complete until the imputation stage of the pipeline. We therefore keep them as `float64` for now rather than forcing an integer type prematurely, the conversion to integer (`Int64`) will happen once every value is filled, in the imputation notebook. 

### 4.3 Data Validation

Before exporting the cleaned datasets, we run three checks: uniqueness of `individual_id` in the Fronds dataset, referential integrity between both datasets, and consistency between the total sori count per individual (`n_sori_total`) and the row-level sori counts (`n_sori_per_row`).

In [24]:
# individual_id should be unique in Fronds
duplicate_ids = df_fronds_clean['individual_id'].duplicated().sum()
print(f"Duplicate individual_id in Fronds: {duplicate_ids}")

#  Referential integrity between datasets
fronds_ids = set(df_fronds_clean['individual_id'])
sori_ids = set(df_sori_clean['individual_id'])

print(f"IDs in Sori but not in Fronds: {sori_ids - fronds_ids}")
print(f"IDs in Fronds but not in Sori: {fronds_ids - sori_ids}")

# Consistency: n_sori_total vs sum of n_sori_per_row
sori_sum = df_sori_clean.groupby('individual_id')['n_sori_per_row'].sum().rename('n_sori_sum')

df_check = df_fronds_clean[['individual_id', 'n_sori_total']].merge(
    sori_sum, on='individual_id', how='left'
)
df_check['difference'] = df_check['n_sori_total'] - df_check['n_sori_sum']
df_check

Duplicate individual_id in Fronds: 0
IDs in Sori but not in Fronds: set()
IDs in Fronds but not in Sori: set()


,individual_id,n_sori_total,n_sori_sum,difference
0,H01,1656.0,1656.0,0.0
1,H02,1554.0,1554.0,0.0
2,H03,1728.0,1728.0,0.0
3,H04,1826.0,1826.0,0.0
4,H05,1456.0,1456.0,0.0
5,H06,1290.0,1290.0,0.0
6,H07,1590.0,1590.0,0.0
7,H08,644.0,644.0,0.0
8,H09,1296.0,1296.0,0.0
9,H10,1464.0,1464.0,0.0


**Summary of findings**

- `individual_id` is unique in the Fronds dataset (0 duplicates), and referential integrity holds in both directions, every ID in Sori exists in Fronds and vice versa.
- 22 of 24 individuals show an exact match between `n_sori_total` and the sum of `n_sori_per_row`. Two exceptions stand out:  

  - **H15**: `n_sori_total` is `NaN`, and its `n_sori_sum` is `0.0`. Section 3 reported 86 missing values (4.48%) in `n_sori_per_row` across the full Sori dataset, and H15 is the only individual whose row-level sum is exactly `0.0`,  since `groupby().sum()` returns `0` only when *every* value in a group is `NaN`, this strongly suggested that all 86 missing values belonged to H15, meaning this individual had no row-level sori data at all rather than a genuine count of zero (confirmed in Section 4.3.1 below).
  - **H19**: a discrepancy of 3 sori (549 vs. 546) between the total and the row-level sum. Both values are present, so this is not a missing-value issue but a genuine data inconsistency.

#### 4.3.1 Verifying the H15 Missing-Values Hypothesis

Section 3 reported 86 missing values in `n_sori_per_row` across the full Sori dataset. We verify whether all 86 belong to individual H15, as suggested by its `n_sori_sum` of `0.0` in the check above.

Before running the check, it is worth spelling out what each possible outcome would mean:

- If `missing_h15 == 86`: this confirms that *all* of the dataset's missing values are concentrated in H15, meaning H15 has no row-level sori data at all, most likely because it was never measured. This is a strong finding: not "a few values are missing," but "an entire individual is missing."
- If `missing_h15 < 86`: this would mean missing values are also scattered across other individuals, and those individuals would still match in the sum in Section 4.3 above, since a `NaN` does not break the equality unless *every* row for that individual is null (which is precisely when `groupby().sum()` returns `0.0` instead of a partial number).

In [25]:
missing_h15 = df_sori_clean[df_sori_clean['individual_id'] == 'H15']['n_sori_per_row'].isna().sum()
missing_total = df_sori_clean['n_sori_per_row'].isna().sum()

print(f"Missing values in H15: {missing_h15}")
print(f"Missing values in the full dataset: {missing_total}")
print(f"H15 accounts for all missing values: {missing_h15 == missing_total}")

Missing values in H15: 86
Missing values in the full dataset: 86
H15 accounts for all missing values: True


Confirmed: all 86 missing values in `n_sori_per_row` belong to individual H15, which has no row-level sori data at all. This is not a scattered missing-data problem, it is a single individual missing entirely from the row-level measurements. This will need to be handled explicitly in the imputation stage of the pipeline.

#### 4.3.2 Investigating the H19 Discrepancy

We check whether H19 has a gap in its `row_id` sequence, which would indicate a missing row rather than an aggregate counting error. Note that individuals are expected to have a different number of rows from one another, since frond rows varies naturally across individuals, the relevant check here is whether H19's *own* sequence is complete, not how it compares to other individuals.

In [26]:
h19 = df_sori_clean[df_sori_clean['individual_id'] == 'H19'].sort_values('row_id')

expected_rows = set(range(h19['row_id'].min(), h19['row_id'].max() + 1))
actual_rows = set(h19['row_id'])
missing_rows = expected_rows - actual_rows

print(f"H19 row_id range: {h19['row_id'].min()} to {h19['row_id'].max()}")
print(f"Number of rows present: {len(h19)}")
print(f"Missing row_id values: {missing_rows if missing_rows else 'None'}")

H19 row_id range: 1 to 88
Number of rows present: 88
Missing row_id values: None


H19's `row_id` sequence is complete, from 1 to 88, with all 88 rows present and no gaps. This rules out a missing row as the explanation for the discrepancy between `n_sori_total` (549) and the sum of `n_sori_per_row` (546).

The row-level data was verified against the original field records by the author, who conducted the data collection. This confirms that 546 is the correct total for H19, and that `n_sori_total` (549) contains a data entry error, most likely introduced when the aggregate value was recorded separately from the row-level counts. This is corrected in Section 4.3.3 below.

Even without direct access to the field records, trusting `n_sori_per_row` over `n_sori_total` would likely have been the more defensible default: the row-level sum is built from 88 individually recorded observations, while the aggregate total is a single manually recorded value, generally more exposed to transcription error than a sum derived from granular data.

#### 4.3.3 Correcting the H19 Total

Based on the verification above, we correct `n_sori_total` for H19 to match the confirmed row-level sum.

In [27]:
df_fronds_clean.loc[df_fronds_clean['individual_id'] == 'H19', 'n_sori_total'] = 546

# Confirm the correction
df_fronds_clean[df_fronds_clean['individual_id'] == 'H19']

,individual_id,frond_length_cm,frond_width_cm,n_sori_total,leaf_area_cm2
18,H19,56.4,5.6,546.0,187.6


`n_sori_total` for H19 has been corrected from 549 to 546. This is the only manual correction applied in this notebook. Unlike the rest of the pipeline, which relies exclusively on values present in the raw data, this correction is based on direct verification against the original field records rather than a statistical inference, it is documented here explicitly for transparency and reproducibility.

## 5. Export Clean Datasets

We export both cleaned datasets to `data/processed/`, so the next notebook in the pipeline can load them directly without repeating the cleaning and validation steps performed here.

In [28]:
df_fronds_clean.to_csv("../data/processed/fronds_clean.csv", index=False)
df_sori_clean.to_csv("../data/processed/sori_clean.csv", index=False)

print(f"df_fronds_clean exported: {df_fronds_clean.shape[0]} rows × {df_fronds_clean.shape[1]} columns")
print(f"df_sori_clean exported: {df_sori_clean.shape[0]} rows × {df_sori_clean.shape[1]} columns")

df_fronds_clean exported: 24 rows × 5 columns
df_sori_clean exported: 1921 rows × 3 columns


Both datasets have been cleaned, standardized, validated, and exported. Missing values remain in both datasets for individual H15: it has no row-level records in Sori (`n_sori_per_row`) and no total in Fronds (`n_sori_total`) either. These will be addressed in the next notebook of the pipeline, focused on imputation.

